# 미세먼지 기사가 많았던 날, 실제로 미세먼지가 짙었을까?
준비물: 빅카인즈에서 내려받은 '미세먼지' 기사 엑셀(2022-04-01 ~ 2022-12-31), 수업 파일 `data_air_2022.csv`
→ 두 파일을 이 노트북과 같은 폴더에 두세요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

## 코드 ① 날짜별 기사 수 세고, 미세먼지와 합치기

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 노트북 파일과 같은 작업 폴더에 수업 CSV와 빅카인즈 엑셀 파일을 둡니다.
folder = Path.cwd()
files = sorted(folder.glob("*NewsResult_*.xlsx"))
if not files:
    raise FileNotFoundError(
        f"빅카인즈 원본 엑셀(*NewsResult_*.xlsx)이 없습니다: {folder}\n"
        "수업에서 내려받은 파일을 해당 폴더에 넣어 주세요."
    )
news = pd.concat([pd.read_excel(f) for f in files], ignore_index=True)
air_path = folder / "data_air_2022.csv"
if not air_path.is_file():
    raise FileNotFoundError(f"미세먼지 원본 CSV 파일이 없습니다: {air_path}")

# 수업 CSV 인코딩이 UTF-8 또는 CP949인지 확인하고 적합한 방식으로 읽습니다.
last_error = None
for encoding in ("utf-8-sig", "cp949"):
    try:
        air = pd.read_csv(air_path, encoding=encoding)
        print("CSV 읽기 성공, 인코딩:", encoding)
        break
    except UnicodeDecodeError as exc:
        last_error = exc
else:
    raise UnicodeError(
        "UTF-8-SIG 및 CP949 모두 해독에 실패했습니다. 원본 CSV 형식을 확인하세요."
    ) from last_error

required_news = {"일자"}
required_air = {"day", "pm10"}
if not required_news.issubset(news.columns):
    raise ValueError(f"기사 파일에 필요한 열 누락: {required_news - set(news.columns)}")
if not required_air.issubset(air.columns):
    raise ValueError(f"대기오염 CSV에 필요한 열 누락: {required_air - set(air.columns)}")
air["pm10"] = pd.to_numeric(air["pm10"], errors="coerce")
print("기사 원본 수:", len(news), "대기질 원본 수:", len(air))
print("기사 열:", news.columns.tolist())
print("대기질 열:", air.columns.tolist())


In [ ]:
print(news.shape) 
print(air.shape) 

In [ ]:
# 기간을 나눠 받은 파일도 위에서 한 번에 합쳐져요
# '일자' 20220405 → 날짜로
news['day'] = pd.to_datetime(news['일자'].astype(str), format='%Y%m%d')
daily = news.groupby('day').size().reset_index(name='기사수')

air['day'] = pd.to_datetime(air['day'])
df = pd.merge(air[['day', 'pm10']], daily, on='day', how='left')
df['기사수'] = df['기사수'].fillna(0)    # 기사 없던 날 = 0
df.head()

## 코드 ② 위아래 두 칸, 같은 날짜축으로 그리기 (축을 겹치지 않아요)

In [ ]:
fig, (a, b) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a.bar(df['day'], df['기사수'])
a.set_ylabel('기사 수(건)')
b.plot(df['day'], df['pm10'])
b.set_ylabel('PM10(㎍/㎥)')
plt.show()

## 코드 ③ 숫자로 확인하고, 어긋난 날 찾기

In [ ]:
print(df['pm10'].corr(df['기사수']))    # 상관계수 r

In [ ]:
# 농도는 낮은데(하위 50%) 기사는 많은(상위 10%) 날
low  = df['pm10'] < df['pm10'].median()
many = df['기사수'] > df['기사수'].quantile(0.9)
df[low & many].sort_values('기사수', ascending=False)

In [ ]:
# 농도는 튀는데(이상치) 기사는 적은 날
q1, q3 = df['pm10'].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
df[(df['pm10'] > upper) & (df['기사수'] < df['기사수'].median())]

찾은 날짜로 빅카인즈에서 그날 기사 **제목**을 읽어 보세요. 정책 발표? 해외 뉴스? → 나만의 탐사 기획서 한 장으로 정리